## 02 — The Comparison

This is the final notebook.

We put both approaches side by side: the system we built across seven modules and the one command that replaces most of it. The goal is not to show that `tippecanoe` is better — it obviously is for production use — but to show exactly **what work it is doing**, because we built every one of those pieces ourselves.

## System Comparison — Architecture

```
OUR SYSTEM                                  TIPPECANOE + TILE CLIENT
─────────────────────────────────────       ──────────────────────────────────────
ne_10m_railroads.geojson (40 MB)            ne_10m_railroads.geojson (40 MB)
       │                                           │
       ▼                                           ▼
Module 02: simplify at 4 epsilons           tippecanoe (one command, ~30 seconds)
  → 4 GeoJSON output files                        │
       │                                           ▼
       ▼                                    railroads.pmtiles (~3 MB)
Module 04: build 4 GridIndex objects               │
  (startup: ~5 seconds)                            ▼
       │                               tile client (browser or localtileserver)
       ▼                                 fetches only visible tiles on demand
Module 05: get_lod(zoom) selects index
       │
       ▼
Module 03: bbox cull within selected index
       │
       ▼
GeoJSON layer (re-sent every pan/zoom)
```

## System Comparison — Numbers

In [1]:
from pathlib import Path
import json
import time


def find_data_file(filename):
    """Find data/filename from the notebook folder or any parent folder."""
    cwd = Path.cwd().resolve()

    for base in [cwd] + list(cwd.parents):
        candidate = base / "data" / filename
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        f"Could not find data/{filename}. Run this notebook from inside your project folder "
        "or keep the data folder inside the project."
    )


raw = find_data_file("ne_10m_railroads.geojson")
data_dir = raw.parent
pmtiles = data_dir / "railroads.pmtiles"

# SINGLE-FILE VERSION:
# This notebook uses only data/ne_10m_railroads.geojson.
# It does NOT require ../../data/lod/railroads_coarse.geojson or any other LOD files.
raw_mb = raw.stat().st_size / 1_000_000
pm_mb = pmtiles.stat().st_size / 1_000_000 if pmtiles.exists() else None

t0 = time.perf_counter()
with open(raw) as f:
    raw_data = json.load(f)
load_time = time.perf_counter() - t0
features = raw_data["features"]


def count_points_in_coords(coords):
    """Count points in LineString or MultiLineString coordinates."""
    if not coords:
        return 0
    if isinstance(coords[0], (int, float)):
        return 1
    return sum(count_points_in_coords(part) for part in coords)


total_pts = sum(count_points_in_coords(f["geometry"]["coordinates"]) for f in features)

print("Storage comparison:")
print(f"  Raw GeoJSON:                         {raw_mb:.1f} MB")
print(f"  Our notebook version, single file:   {raw_mb:.1f} MB")
if pm_mb is not None:
    print(f"  tippecanoe PMTiles:                  {pm_mb:.1f} MB")
else:
    print("  tippecanoe PMTiles:                  (run Notebook 01 first)")

print()
print("Single-file data loaded:")
print(f"  File:        {raw.name}")
print(f"  Features:    {len(features):,}")
print(f"  Total pts:   {total_pts:,}")
print(f"  Load time:   {load_time:.3f}s")

print()
print("Runtime comparison:")
print("  Our startup:               loads the one raw GeoJSON file and builds any in-memory index used by the viewer")
print("  Our per-query time:         fast after indexing, but the full raw file still had to load first")
print("  Tile client startup:        ~0s because it lazily fetches only visible tiles")
print("  Tile fetch over network:    usually small per tile instead of downloading the full GeoJSON")
print("  Tile fetch local:           very fast if railroads.pmtiles exists locally")


Storage comparison:
  Raw GeoJSON:                         39.6 MB
  Our notebook version, single file:   39.6 MB
  tippecanoe PMTiles:                  (run Notebook 01 first)

Single-file data loaded:
  File:        ne_10m_railroads.geojson
  Features:    25,413
  Total pts:   1,396,480
  Load time:   1.017s

Runtime comparison:
  Our startup:               loads the one raw GeoJSON file and builds any in-memory index used by the viewer
  Our per-query time:         fast after indexing, but the full raw file still had to load first
  Tile client startup:        ~0s because it lazily fetches only visible tiles
  Tile fetch over network:    usually small per tile instead of downloading the full GeoJSON
  Tile fetch local:           very fast if railroads.pmtiles exists locally


## What Tippecanoe Automated — Specifically

Now name each thing precisely — because you built it:

**1. Multi-resolution simplification (our Module 02)**
tippecanoe applies a tolerance appropriate for each zoom level automatically. You do not choose 4 epsilons — you choose one `--simplification` value and it scales it per zoom.

**2. Spatial bucketing (our Module 04)**
Tiles are the index. There is no separate grid index to build — the tile `(z, x, y)` address is the bucket. Features are pre-assigned to tiles at generation time.

**3. Viewport culling (our Module 03)**
The client requests only the tiles its viewport covers. No intersection test — irrelevant tiles are never fetched.

**4. LOD switching (our Module 05)**
The tile URL includes the zoom level (`/{z}/`). The client naturally requests tiles at the right zoom. No decision function needed.

**5. Binary encoding**
We never built this — our GeoJSON is text. Tippecanoe outputs MVT binary with integer coordinates. ~5× smaller and faster to parse.

**6. Streaming / lazy loading**
We never built this either. Tiles are fetched on demand; unused regions are never touched.

## What Tippecanoe Does NOT Do

Tippecanoe is a **preprocessing pipeline** — it generates static files. It does not:

- Serve tiles dynamically (you still need a static host, CDN, or `localtileserver`)
- Filter features at query time based on user input
- Handle real-time data updates
- Decide which features to show based on screen density or user preferences at render time

For live, user-driven filtering (e.g., "show only electrified railways"), a tile system either:
- Pre-generates multiple tile sets (one per filter combination)
- Sends all attributes in the tile and lets the client filter at render time (style expressions)
- Uses a dynamic tile server that queries a database per request

None of these are trivial. Our handbuilt system could add real-time filtering in ten lines.

## The Quote That Started This

> *"Build the smallest version that teaches the idea. Borrow the version that survives the real world."*

You built the smallest version. You know:
- What Douglas-Peucker does and why epsilon matters
- Why spatial indexes exist and what they trade off
- Why bounding box culling is O(n) without an index
- Why the tile coordinate scheme is itself a spatial index
- Why binary encoding is worth the complexity

When you use `tippecanoe` from now on, you can read its flags without guessing. When it produces unexpected output, you can reason about why. When a colleague says "we should just use vector tiles" without understanding the tradeoffs, you can ask the right questions.

That is not the same as having typed `tippecanoe` once.

## Exercise A

Write a one-page (or ~15 bullet point) technical comparison of the two systems. Cover:
- Build time
- Storage footprint
- Startup cost for the end user
- Per-request data transfer
- Support for real-time filtering
- Complexity to maintain
- What you would use for a class project vs. a production application

Write it in the cell below as markdown.

**Exercise A — Technical Comparison**

- Our handbuilt system starts with one raw Natural Earth railroad GeoJSON file and turns it into four simplified GeoJSON files.
- The tippecanoe system starts with the same raw GeoJSON file but turns it into a vector-tile pyramid stored in one PMTiles or MBTiles file.
- Build time for our system is spread across several steps: simplifying features, writing LOD files, calculating bounding boxes, building grid indexes, and wiring a viewer.
- Build time for tippecanoe is mostly one command, so the pipeline is easier to repeat and less likely to break from hand-coded logic.
- Our storage footprint is larger because GeoJSON stores coordinates as readable text and duplicates the world into multiple LOD files.
- Tippecanoe stores tile geometry in a compact binary vector-tile format, so the final output is usually much smaller and faster to parse.
- Our startup cost for the user is high because the notebook loads the LOD files and builds indexes before the map is fully useful.
- A tile client has a much lower startup cost because it fetches only the tiles visible in the current map view.
- Our per-request transfer is inefficient for a remote user because the full files must exist locally or be loaded before the index can help.
- A vector-tile workflow transfers only the needed `{z}/{x}/{y}` tiles, so panning and zooming can stay lightweight.
- Our system is easier to understand for a class project because every component was built directly: simplification, culling, indexing, LOD selection, and live switching.
- Tippecanoe is better for production because it handles zoom-dependent simplification, tile-size limits, tile indexing, binary encoding, and sparse storage automatically.
- Our system is better for teaching and debugging algorithms because we can inspect every line and change each design decision manually.
- Tippecanoe is worse for real-time user-driven filtering if the filtered data was not already included in the tiles, because removed features cannot be recovered in the browser.
- For a class project I would use the handbuilt system to show understanding; for a production map I would use tippecanoe plus a tile client because it scales better and is much easier to deliver to many users.

## Exercise B

Look up the `--attribute-filter` and `--include` flags in the tippecanoe documentation.

Could you use these to produce a tile set that only includes electrified railroads (`electric` property)? Write the command you would use, and explain what the resulting tile set would and would not be able to show.

In [2]:
# Electrified-only tile set command:
#
# tippecanoe \
#   --output=../../data/railroads_electric.pmtiles \
#   --force \
#   --minimum-zoom=1 \
#   --maximum-zoom=14 \
#   --simplification=10 \
#   --drop-densest-as-needed \
#   --layer=railroads \
#   --include=electric \
#   --include=name \
#   --feature-filter='{"railroads":["in","electric","yes","Y","true","1"]}' \
#   ../../data/ne_10m_railroads.geojson
#
# Explanation:
# --include keeps only selected attributes in the output tiles.
# It does not remove non-electrified railroad features by itself.
# To remove features, use a feature filter such as --feature-filter / -j.
# The resulting tile set would be smaller and would show only railroads whose electric
# property matches the accepted electrified values in the filter.
# It would not be able to show non-electrified railroads later, because those features
# were removed when the tile set was generated.
# If a user wanted to toggle electrified vs. non-electrified railroads in the browser,
# we would need either a full tile set with the electric attribute preserved or two
# separate tile sets generated ahead of time.


## Check Your Understanding

A classmate who skipped Modules 01–06 and came straight to this notebook could run `tippecanoe` and get a working tile set. They would see the flags but not know what they mean.

Name **three specific situations** where their lack of understanding would cost them — where they would make a wrong decision, miss a bug, or be unable to debug a problem — that you would be able to handle.

---

**Check Your Understanding Answer**

1. They might choose the wrong `--maximum-zoom` or `--simplification` value. The tiles would build successfully, but the map could look too jagged, too generalized, or unnecessarily large, and they would not understand that the issue is a zoom-resolution tradeoff.

2. They might use `--drop-densest-as-needed` without realizing that some features can be removed from crowded low-zoom tiles. If a railroad line disappears, they might think the data is broken instead of recognizing that the tile-size rule dropped low-visibility features.

3. They might confuse attribute filtering with feature filtering. For example, `--include=electric` preserves the `electric` field, but it does not create an electrified-only railroad map; to remove non-electrified railroads, they need a feature filter or a pre-filtered input file.

## End of the Data Manager Micro Lessons

You built:
- A simplification algorithm
- A multi-resolution data pipeline
- A spatial intersection test
- A grid-based spatial index
- A zoom-driven data selection system
- A working interactive map viewer
- An informed opinion about when to stop building and borrow instead

The railroad project is where you apply it.